PURPOSE: To understand the data before building.

 Main questions:
 1. Is the data structurally clean?
 2. What does the demand distribution look like?
 3. Is there a trend over time?
 4. Is there seasonality?
 5. Is demand autocorrelated?
 6. Are there unusual demand regimes?
 7. Does demand differ across medicines and regions?
 8. Could inventory constraints affect observed sales?

In [ ]:
# Data Audit + Exploratory Data Analysis

#  LOAD DATA
# ============================================================

# Upload the CSV file from your computer.
from google.colab import files

uploaded = files.upload()


# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu, skew
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)


# Load the dataset
df = pd.read_csv("global_pharmacy_sales_2020_2025_daily_dataset.csv")

In [ ]:

# Convert date column into a proper datetime variable
df["date"] = pd.to_datetime(df["date"])


print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Date range:", df["date"].min(), "to", df["date"].max())

display(df.head())


#  BASIC DATA STRUCTURE
# ============================================================

print("\nColumn names:")
print(df.columns.tolist())


print("\nData types:")
print(df.dtypes)


print("\nUnique values:")
for col in ["region", "country", "category", "medicine", "age_group"]:
    print(f"{col}: {df[col].nunique()}")


# Check the number of unique dates
print("\nUnique dates:", df["date"].nunique())


# DATA QUALITY — MISSING VALUES
# ============================================================

# Count missing values in every column
missing = df.isna().sum().to_frame("missing_count")

# Convert counts into percentages
missing["missing_percent"] = (
    missing["missing_count"] / len(df) * 100
)

print("\nMissingness:")
display(missing)


# DUPLICATES
# ============================================================

# Exact duplicates:
# Every column is identical.
exact_dupes = df.duplicated().sum()


# Logical duplicates: Same date + region + country + medicine + age group.

logical_dupes = df.duplicated(
    subset=[
        "date",
        "region",
        "country",
        "medicine",
        "age_group"
    ],
    keep=False
).sum()


print(f"\nExact duplicate rows: {exact_dupes}")
print(f"Logical duplicate rows: {logical_dupes}")


# INVALID VALUES
# ============================================================

# Check whether important numerical variables contain negative values.
numeric_columns = [
    "units_sold",
    "unit_price",
    "stock_level",
    "expiry_days_remaining"
]


print("\nNegative values:")

for col in numeric_columns:
    negative_count = (df[col] < 0).sum()
    print(f"{col}: {negative_count}")


# Check basic ranges
print("\nNumerical summary:")
display(df[numeric_columns].describe())

In [ ]:
# TARGET DISTRIBUTION — UNITS SOLD
# ============================================================

sales = df["units_sold"]


# Basic statistics
sales_mean = sales.mean()
sales_median = sales.median()
sales_skew = skew(sales)


print("\nUnits Sold Distribution")
print("-----------------------")
print(f"Mean:   {sales_mean:.2f}")
print(f"Median: {sales_median:.2f}")
print(f"Skew:   {sales_skew:.2f}")


# IQR-based outlier detection
q1 = sales.quantile(0.25)
q3 = sales.quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr


outliers = df[
    (df["units_sold"] < lower_bound) |
    (df["units_sold"] > upper_bound)
]


print(f"\nQ1: {q1:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")

print(
    f"IQR outliers: {len(outliers):,} "
    f"({len(outliers) / len(df) * 100:.2f}%)"
)


# Plot the distribution
plt.figure(figsize=(10, 5))

plt.hist(
    sales,
    bins=100
)

plt.title("Distribution of Units Sold")
plt.xlabel("Units Sold")
plt.ylabel("Frequency")

plt.show()


#  DEMAND OVER TIME — OVERALL TREND
# ============================================================

# Aggregate all observations by date.
#
# This gives us the total observed utilization across
# medicines and regions for each day.
daily_demand = (
    df.groupby("date")["units_sold"]
      .sum()
      .sort_index()
)


# Calculate a 30-day rolling average.
# The rolling average makes the underlying trend easier
# to see than the noisy daily series.
rolling_30 = daily_demand.rolling(30).mean()


plt.figure(figsize=(14, 6))

plt.plot(
    daily_demand.index,
    daily_demand.values,
    alpha=0.35,
    label="Daily demand"
)

plt.plot(
    rolling_30.index,
    rolling_30.values,
    label="30-day rolling average"
)

plt.title("Overall Observed Demand Over Time")
plt.xlabel("Date")
plt.ylabel("Units Sold")

plt.legend()
plt.show()


#  MONTHLY DEMAND TREND
# ============================================================

# Aggregate demand by month.
monthly_demand = (
    df.set_index("date")
      .resample("MS")["units_sold"]
      .sum()
)


plt.figure(figsize=(14, 5))

plt.plot(
    monthly_demand.index,
    monthly_demand.values
)

plt.title("Monthly Observed Demand")
plt.xlabel("Month")
plt.ylabel("Total Units Sold")

plt.show()


# SEASONALITY
# ============================================================

# Create calendar month variable.
df["month"] = df["date"].dt.month


# Average daily units sold for each calendar month.
monthly_avg = (
    df.groupby("month")["units_sold"]
      .mean()
)


print("\nAverage Units Sold by Calendar Month:")
display(monthly_avg.to_frame("average_units_sold"))


# Plot the monthly seasonal pattern.
plt.figure(figsize=(10, 5))

plt.bar(
    monthly_avg.index,
    monthly_avg.values
)

plt.title("Average Units Sold by Calendar Month")
plt.xlabel("Calendar Month")
plt.ylabel("Average Units Sold")

plt.xticks(range(1, 13))

plt.show()


# AUTOCORRELATION — ACF
# ============================================================

# Aggregate to a single daily demand series.
#
# ACF tells us whether current demand is related to
# demand at previous time points.
#
# This is useful because forecasting models often rely
# on lagged demand information.

ts = daily_demand.dropna()


plt.figure(figsize=(12, 5))

plot_acf(
    ts,
    lags=60
)

plt.title("Autocorrelation of Daily Observed Demand")

plt.show()


# PARTIAL AUTOCORRELATION — PACF
# ============================================================

# PACF shows the relationship between current demand and
# a particular lag after accounting for shorter lags.

plt.figure(figsize=(12, 5))

plot_pacf(
    ts,
    lags=60,
    method="ywm"
)

plt.title("Partial Autocorrelation of Daily Observed Demand")

plt.show()


#  COVID / DEMAND REGIME ANALYSIS
# ============================================================

# Compare observed utilization during COVID-flagged
# and non-COVID periods.

non_covid = df.loc[
    df["covid_flag"] == 0,
    "units_sold"
]

covid = df.loc[
    df["covid_flag"] == 1,
    "units_sold"
]


# Compare means
non_covid_mean = non_covid.mean()
covid_mean = covid.mean()


# Calculate percentage difference
covid_change_pct = (
    (covid_mean - non_covid_mean)
    / non_covid_mean
) * 100


# Mann-Whitney test
#
# This tests whether the distributions differ.
stat, p_value = mannwhitneyu(
    non_covid,
    covid,
    alternative="two-sided"
)


print("\nCOVID-period Demand Regime Analysis")
print("------------------------------------")

print(f"Non-COVID mean: {non_covid_mean:.2f}")
print(f"COVID mean:     {covid_mean:.2f}")
print(f"Mean change:    {covid_change_pct:.2f}%")
print(f"Mann-Whitney p-value: {p_value:.2e}")


# Compare the distributions visually.
plt.figure(figsize=(8, 5))

plt.boxplot(
    [non_covid, covid],
    tick_labels=["Non-COVID", "COVID"]
)
plt.title("Units Sold: COVID vs Non-COVID Periods")
plt.ylabel("Units Sold")

plt.show()


# HIGH-DEMAND OBSERVATIONS AND COVID
# ============================================================

# Check whether the IQR outliers are concentrated
# in COVID periods.

outlier_covid_rate = (
    outliers["covid_flag"] == 1
).mean() * 100


non_outliers = df.drop(outliers.index)


non_outlier_covid_rate = (
    non_outliers["covid_flag"] == 1
).mean() * 100


print("\nCOVID share:")
print(
    f"Among IQR outliers:     {outlier_covid_rate:.2f}%"
)

print(
    f"Among non-outliers:     {non_outlier_covid_rate:.2f}%"
)


#  STOCK LEVEL / POTENTIAL INVENTORY CONSTRAINT
# ============================================================

# Examine the relationship between observed units sold
# and stock level.
#
# IMPORTANT:
# We do NOT assume that stock_level represents true beginning-of-day or end-of-day inventory.
#
# Therefore this section identifies a potential issue
# rather than claiming that demand is or is not censored.

correlation = df["units_sold"].corr(
    df["stock_level"]
)


# Percentage of observations with zero stock.
zero_stock_pct = (
    df["stock_level"] == 0
).mean() * 100


# Compare demand when stock is zero vs positive.
zero_stock_demand = df.loc[
    df["stock_level"] == 0,
    "units_sold"
]

positive_stock_demand = df.loc[
    df["stock_level"] > 0,
    "units_sold"
]


print("\nStock-Level Investigation")
print("-------------------------")

print(
    f"Correlation between units_sold and stock_level: "
    f"{correlation:.3f}"
)

print(
    f"Rows with zero stock: "
    f"{zero_stock_pct:.2f}%"
)

print(
    f"Mean units sold when stock = 0: "
    f"{zero_stock_demand.mean():.2f}"
)

print(
    f"Mean units sold when stock > 0: "
    f"{positive_stock_demand.mean():.2f}"
)


# Plot relationship using a sample to avoid an overcrowded chart.
plot_sample = df.sample(
    min(10000, len(df)),
    random_state=42
)


plt.figure(figsize=(10, 6))

plt.scatter(
    plot_sample["stock_level"],
    plot_sample["units_sold"],
    alpha=0.25
)

plt.title("Observed Units Sold vs Stock Level")
plt.xlabel("Stock Level")
plt.ylabel("Units Sold")

plt.show()


# REGIONAL DEMAND HETEROGENEITY
# ============================================================

# Total demand by region.
region_demand = (
    df.groupby("region")["units_sold"]
      .sum()
      .sort_values(ascending=False)
)


print("\nTotal Observed Demand by Region:")
display(region_demand.to_frame("total_units_sold"))


plt.figure(figsize=(10, 5))

plt.bar(
    region_demand.index,
    region_demand.values
)

plt.title("Total Observed Demand by Region")
plt.xlabel("Region")
plt.ylabel("Total Units Sold")

plt.xticks(rotation=45)

plt.show()


# MEDICINE-LEVEL DEMAND HETEROGENEITY
# ============================================================

medicine_demand = (
    df.groupby("medicine")["units_sold"]
      .sum()
      .sort_values(ascending=False)
)


print("\nTotal Observed Demand by Medicine:")
display(
    medicine_demand.to_frame("total_units_sold")
)


plt.figure(figsize=(10, 5))

plt.bar(
    medicine_demand.index,
    medicine_demand.values
)

plt.title("Total Observed Demand by Medicine")
plt.xlabel("Medicine")
plt.ylabel("Total Units Sold")

plt.xticks(rotation=45)

plt.show()


#  SERIES COVERAGE / TIME GAPS
# ============================================================

# The eventual forecasting grain is:
#
# medicine × region × month
#
# We therefore check whether every medicine-region series
# has the same monthly coverage.

df["year_month"] = df["date"].dt.to_period("M")


series_month_counts = (
    df.groupby(
        ["medicine", "region"]
    )["year_month"]
    .nunique()
)


print("\nMonthly observations per medicine-region series:")
display(series_month_counts.describe())


# Identify series with fewer months than the maximum.
max_months = series_month_counts.max()

incomplete_series = series_month_counts[
    series_month_counts < max_months
]


print(
    f"\nMaximum number of observed months per series: "
    f"{max_months}"
)

print(
    f"Medicine-region series with fewer months: "
    f"{len(incomplete_series)}"
)


if len(incomplete_series) > 0:
    display(
        incomplete_series.to_frame(
            "observed_month_count"
        ).head(20)
    )


# MEDICINE → CATEGORY CONSISTENCY
# ============================================================

# A medicine should ideally belong to one category.
#
# If the same medicine appears under multiple categories,
# this needs to be understood before modeling.

category_counts = (
    df.groupby("medicine")["category"]
      .nunique()
)


inconsistent_medicines = category_counts[
    category_counts > 1
]


print("\nMedicine-category consistency")
print("-----------------------------")

print(
    f"Medicines mapped to more than one category: "
    f"{len(inconsistent_medicines)}"
)


if len(inconsistent_medicines) > 0:
    display(inconsistent_medicines)


#  BASIC DATA QUALITY SUMMARY
# ============================================================

# Summarize important quality checks in one place.

negative_units = (
    df["units_sold"] < 0
).sum()

negative_price = (
    df["unit_price"] < 0
).sum()

negative_stock = (
    df["stock_level"] < 0
).sum()

negative_expiry = (
    df["expiry_days_remaining"] < 0
).sum()


print("\nData Quality Summary")
print("--------------------")

print(f"Missing cells: {df.isna().sum().sum():,}")
print(f"Exact duplicates: {exact_dupes:,}")
print(f"Logical duplicate rows: {logical_dupes:,}")
print(f"Negative units_sold: {negative_units:,}")
print(f"Negative unit_price: {negative_price:,}")
print(f"Negative stock_level: {negative_stock:,}")
print(f"Negative expiry_days_remaining: {negative_expiry:,}")


# FINAL DATA QUALITY & ANALYTICAL DECISION LOG
# ============================================================



decision_log = pd.DataFrame([

    {
        "Finding": "Strong right skew in units_sold",
        "Decision": "Retain",
        "Reason":
            "Demand is strongly right-skewed. "
            "Extreme values are not automatically treated "
            "as data errors."
    },

    {
        "Finding":
            "High-demand observations are concentrated "
            "in COVID-flagged periods",
        "Decision":
            "Retain and treat COVID as a potential demand regime",
        "Reason":
            f"Mean observed units_sold differed by "
            f"{covid_change_pct:.1f}% between COVID and "
            f"non-COVID periods, with Mann-Whitney "
            f"p={p_value:.2e}. The observations may represent "
            f"a different demand regime rather than erroneous data."
    },

    {
        "Finding":
            "Observed units_sold has a measurable relationship "
            "with stock_level",
        "Decision":
            "Flag as a potential inventory constraint",
        "Reason":
            f"Correlation = {correlation:.2f} and "
            f"{zero_stock_pct:.1f}% of observations have zero stock. "
            f"Inventory timing and replenishment mechanics are not "
            f"sufficiently documented to conclude whether observed "
            f"sales represent unconstrained demand."
    },

    {
        "Finding":
            f"{exact_dupes} exact duplicate rows",
        "Decision":
            "Remove duplicates during data staging",
        "Reason":
            "Identical records can lead to double counting "
            "and should not be treated as independent observations."
    },

    {
        "Finding":
            f"{logical_dupes} rows involved in logical duplicate check",
        "Decision":
            "Investigate before aggregation",
        "Reason":
            "Multiple records at the expected daily observation "
            "grain may indicate duplicate or unexpected records."
    },

    {
        "Finding":
            "Missing values",
        "Decision":
            "Handle explicitly during data staging",
        "Reason":
            "Missing values should be investigated by variable "
            "rather than blindly imputed."
    },

    {
        "Finding":
            "Potential missing months in medicine-region series",
        "Decision":
            "Do not automatically replace missing periods with zero",
        "Reason":
            "Absence of a record does not necessarily mean zero demand. "
            "The monthly analytical table will explicitly check series coverage."
    },

    {
        "Finding":
            "Temporal dependence detected through time-series diagnostics",
        "Decision":
            "Use time-aware feature engineering and validation",
        "Reason":
            "Autocorrelation supports the use of historical lagged "
            "information and time-series-aware model evaluation."
    },

    {
        "Finding":
            "Seasonal variation across calendar months",
        "Decision":
            "Retain seasonal information",
        "Reason":
            "Calendar-month patterns may contain useful predictive "
            "information and justify seasonal baselines."
    },

    {
        "Finding":
            "First observed date for medicine-region series",
        "Decision":
            "Treat as launch_proxy_date",
        "Reason":
            "The dataset does not provide verified commercial launch "
            "events. First observation therefore represents dataset "
            "availability rather than confirmed market launch."
    },

    {
        "Finding":
            "Demand varies across medicines and regions",
        "Decision":
            "Forecast at medicine-region analytical grain",
        "Reason":
            "Aggregating all products and geographies into one global "
            "series would hide meaningful cross-sectional differences."
    }

])


print("\n" + "=" * 80)
print("DATA QUALITY & ANALYTICAL DECISION LOG")
print("=" * 80)

display(decision_log)


# Save the decision log for later use.
decision_log.to_csv(
    "data_quality_decision_log.csv",
    index=False
)


print(
    "\nDecision log saved as "
    "data_quality_decision_log.csv"
)


# CONCLUSION


print("\n" + "=" * 80)
print("KEY CONCLUSIONS")
print("=" * 80)

print("""
1. The target is strongly right-skewed, so extreme observations
   should not automatically be removed.

2. Demand behavior differs between COVID and non-COVID periods,suggesting that COVID may represent a different demand regime.

3. The data contains temporal structure through trend, seasonality,
   and autocorrelation, supporting time-series forecasting methods.

4. Demand varies across medicines and regions, supporting the use
   of a medicine-region analytical forecasting grain.

5. Stock level shows a relationship with observed utilization.
   However, the available data does not establish the exact timing
   of inventory measurements or replenishment. Therefore observed
   utilization should not automatically be interpreted as
   unconstrained latent demand.

6. Missing time periods will not automatically be converted to zero demand.

7. First observed date will be treated as a launch proxy rather than a verified commercial launch date.

8. These decisions will be carried further where the analytical dataset and forecasting methodology are built.
""")